# FvCB

The FvCB model, also known as the Farquhar, von Caemmerer, and Berry model, is a widely used theoretical framework in plant biology, first conceptualized in 1980. It provides a simplistic view of C3 photosynthesis and is named after its primary contributors: Graham D. Farquhar, Susanne von Caemmerer, and Joseph A. Berry. 


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import brentq

from mxlmodels import get_fvcb

## Versions

As the model has become a staple in steady-state photosynthesis modeling, it has undergone several revisions and updates. Here we include the original [1980 version](https://doi.org/10.1007/BF00386231) and the revised [2025 version](https://doi.org/10.1093/insilicoplants/diaf014) by Lochocki, that explains the correct min-W version of the model.

## Figure recreation

## Original Figures

### Figure 4

In [ ]:
def calculate_quantum_yield(pco2, po2, T, i_low, i_high):
    """Calculates the apparent quantum yield by finding the slope between two irradiances."""
    A_low = get_fvcb(pco2=pco2, po2=po2, T=T, I=i_low)["A"]

    A_high = get_fvcb(pco2=pco2, po2=po2, T=T, I=i_high)["A"]

    return (A_high - A_low) / (i_high - i_low)


fig4_res = pd.DataFrame(
    {
        "pco2": np.linspace(1, 450, 100),
        "yield_210": [calculate_quantum_yield(pco2=c, po2=210, T=298, i_low=50, i_high=150) for c in np.linspace(1, 450, 100)],
        "yield_10": [calculate_quantum_yield(pco2=c, po2=10, T=298, i_low=50, i_high=150) for c in np.linspace(1, 450, 100)],
    }
)

fig4_res = fig4_res.set_index("pco2")

In [ ]:
fig4, ax = plt.subplots(
    figsize=(5, 4))

ax.plot(fig4_res["yield_210"], color="black")
ax.plot(fig4_res["yield_10"], color="black")

ax.set_xlim(0, 430)
ax.set_xlabel("Intercellular CO2, C (µbar)")
ax.set_xticks(np.linspace(0, 400, 5))

ax.set_ylim(-0.02, 0.08)
ax.set_ylabel("Quantum yield (mol CO2 / mol photons)")

text_xval = 150
text_yvals = fig4_res.iloc[fig4_res.index.get_indexer([text_xval], method="nearest")[0]]
distance = 0.005
ax.text(
    x=text_xval,
    y=text_yvals["yield_210"] - distance,
    s="O = 210",
)
ax.text(
    x=text_xval,
    y=text_yvals["yield_10"] - distance,
    s="O = 10 mbar",
)

plt.tight_layout()

### Figure 5

In [ ]:
t_array = np.linspace(0, 40, 100)

fig5_res = pd.DataFrame(
    {
        "T": t_array,
        "yield_660_10": [calculate_quantum_yield(pco2=660, po2=10, T=t, i_low=0, i_high=10) for t in t_array + 273],
        "yield_230_10": [calculate_quantum_yield(pco2=230, po2=210, T=t, i_low=0, i_high=10) for t in t_array + 273],
    }
)

fig5_res = fig5_res.set_index("T")

In [ ]:
fig5, ax = plt.subplots(figsize=(4, 5))

ax.plot(fig5_res["yield_660_10"], color="black")
ax.plot(fig5_res["yield_230_10"], color="black")

ax.set_xlim(0, 40)
ax.set_xlabel("T (C)")

ax.set_ylabel("Quantum yield (mol CO2 / Einstein)")
ax.set_ylim(0, 0.1)

text_xval = 15
text_yvals = fig5_res.iloc[fig5_res.index.get_indexer([text_xval], method="nearest")[0]]
distance = 0.0025
ax.text(
    x=text_xval,
    y=text_yvals["yield_660_10"] + distance,
    s="C = 660 µbar",
    va="bottom",
    ha="center"
)
ax.text(
    x=text_xval,
    y=text_yvals["yield_660_10"] - distance,
    s="O = 10 mbar",
    va="top",
    ha="center"
)
ax.text(
    x=text_xval,
    y=text_yvals["yield_230_10"] + distance,
    s="C = 230",
    va="bottom",
    ha="center"
)
ax.text(
    x=text_xval,
    y=text_yvals["yield_230_10"] - distance,
    s="O = 10",
    va="top",
    ha="center"
)

plt.tight_layout()

### Figure 6

In [ ]:
fig6_res = pd.DataFrame(
    {
        "T": np.linspace(0, 40, 100) + 273,
    }
)

def get_gamma(po2: float, I: float, T: float):
    def get_assimilation(c):
                return get_fvcb(pco2=c, po2=po2, I=I, T=T)["A"]

    return brentq(get_assimilation, 1, 230)

fig6_res["gamma_100"] = fig6_res["T"].apply(lambda x: get_gamma(po2=210, I=100, T=x))
fig6_res["gamma_1000"] = fig6_res["T"].apply(lambda x: get_gamma(po2=210, I=1000, T=x))

fig6_res = fig6_res.set_index("T")
fig6_res.index -= 273

In [ ]:
fig6, ax = plt.subplots(figsize=(4,3.5))

ax.plot(fig6_res["gamma_100"], color="black")
ax.plot(fig6_res["gamma_1000"], color="black")

ax.set_xlim(0, 40)
ax.set_xlabel("T (C)")

ax.set_ylim(0, 170)
ax.set_yticks(np.linspace(0, 160, 5))
ax.set_ylabel("CO2 compensation point, $\\Gamma$ (µbar)")

text_xval = 29
text_yvals = fig6_res.iloc[fig6_res.index.get_indexer([text_xval], method="nearest")[0]]
distance = 2
ax.text(
    x=text_xval,
    y=text_yvals["gamma_100"] + distance,
    s="I = 100 µE m⁻² s⁻¹",
    va="bottom",
    ha="right"
)
ax.text(
    x=text_xval,
    y=text_yvals["gamma_1000"] - distance,
    s="I = 1000",
    va="top",
    ha="left"
)

ax.text(
    x=10,
    y=140,
    s="O=210 mbar",
    va="top",
    ha="left"
)

plt.tight_layout()

### Figure 7

In [ ]:
fig7_res = pd.DataFrame(
    {
        "pco2": np.linspace(1, 450, 100),
    }
)

for key in ["A", "Vc", "Vo"]:
    fig7_res[key] = fig7_res["pco2"].apply(lambda x: get_fvcb(pco2=x)[key])

fig7_res["A_10"] = fig7_res["pco2"].apply(lambda x: get_fvcb(pco2=x, po2=10)["A"])
fig7_res = fig7_res.set_index("pco2")

In [ ]:
fig7, ax = plt.subplots(figsize=(4,7))

ax.plot(fig7_res["A"], color="black")
ax.plot(fig7_res["Vc"], color="black")
ax.plot(0.5*fig7_res["Vo"], color="black")
ax.plot(fig7_res["A_10"], color="black", ls="--")

ax.set_xlim(0, 450)
ax.set_xlabel("Intercellular CO₂, C (µbar)")

ax.set_ylim(-5, 30)
ax.set_ylabel("CO2 fluxes (μmol m⁻² s⁻¹)")

text_xval = 150
text_yvals = fig7_res.iloc[fig7_res.index.get_indexer([text_xval], method="nearest")[0]]
distance = 2
ax.text(
    x=text_xval,
    y=text_yvals["A_10"] + distance,
    s="O=10 mbar",
    va="bottom",
    ha="right"
)
ax.text(
    x=text_xval,
    y=text_yvals["A"] - distance,
    s="O=10 mbar",
    va="top",
    ha="left"
)

text_xval = 250
text_yvals = fig7_res.iloc[fig7_res.index.get_indexer([text_xval], method="nearest")[0]]
distance = 1
ax.text(
    x=text_xval,
    y=text_yvals["A"] + distance,
    s="A",
    va="bottom",
    ha="center"
)
ax.text(
    x=text_xval,
    y=text_yvals["Vc"] + distance,
    s="Vc",
    va="bottom",
    ha="center"
)
ax.text(
    x=text_xval,
    y=text_yvals["Vo"] * 0.5 + distance,
    s="0.5 $\\times$Vo",
    va="bottom",
    ha="center"
)

text_xval = 200
text_yvals = fig7_res.iloc[fig7_res.index.get_indexer([text_xval], method="nearest")[0]]
distance = 1
ax.text(
    x=text_xval,
    y=text_yvals["A_10"] + distance,
    s="A",
    va="bottom",
    ha="center"
)

plt.tight_layout()

### Figure 8

In [ ]:
t_array = np.linspace(0, 50, 200)

fig8_res = pd.DataFrame(
    {
        "T": t_array,
    }
)

for pco2 in [330, 230, 165, 110, 55]:
    fig8_res[f"A_{pco2}"] = [
        get_fvcb(pco2=pco2, po2=210, T=t + 273, I=700)["A"] for t in t_array
    ]
    
fig8_res = fig8_res.set_index("T")

In [ ]:
fig8, ax = plt.subplots(figsize=(3,5))

for pco2 in [330, 230, 165, 110, 55]:
    ax.plot(fig8_res[f"A_{pco2}"], color="black", label=f"C={pco2} µbar")

ax.set_xlim(0, 45)
ax.set_xticks(np.linspace(0, 40, 5))
ax.set_xlabel("T (C)")

ax.set_ylim(-5, 30)
ax.set_yticks(np.linspace(0, 25, 6))
ax.set_ylabel("Assimilation rate, A (μmol m⁻² s⁻¹)")

text_xval = 30
text_yvals = fig8_res.iloc[fig8_res.index.get_indexer([text_xval], method="nearest")[0]]
distance = 1
for pco2 in [330, 230, 165, 110, 55]:
    ax.text(
        x=text_xval,
        y=text_yvals[f"A_{pco2}"] + distance,
        s=f"{pco2}" if pco2 != 330 else f"C= {pco2} µbar",
        va="bottom",
        ha="center"
    )
    
ax.text(
    x=1,
    y=25,
    s="I=700 µE m⁻² s⁻¹\nO=210 mbar",
    va="top",
    ha="left"
)

plt.tight_layout()

### Figure 9

In [ ]:
t_array = np.linspace(0, 50, 200)

fig9_res = pd.DataFrame(
    {
        "T": t_array,
    }
)

for I in [1000, 500, 250, 100, 0]:
    fig9_res[f"A_{I}"] = [
        get_fvcb(pco2=230, po2=210, T=t + 273, I=I)["A"] for t in t_array
    ]
    
fig9_res["A_r0"] = [get_fvcb(pco2=230, po2=210, T=t + 273, I=1000, r_light=0)["A"] for t in t_array]
fig9_res["A_jinf"] = [get_fvcb(pco2=230, po2=210, T=t + 273, I=1000, j_infinite=True)["A"] for t in t_array]
fig9_res["A_r0jinf"] = [get_fvcb(pco2=230, po2=210, T=t + 273, I=1000, r_light=0, j_infinite=True)["A"] for t in t_array]

fig9_res = fig9_res.set_index("T")

In [ ]:
fig9, ax = plt.subplots(figsize=(3,5))

for I in [1000, 500, 250, 100, 0]:
    ax.plot(fig9_res[f"A_{I}"], color="black", label=f"I={I} µmol m⁻² s⁻¹")
    
ax.plot(fig9_res["A_r0"], color="black", ls="dashed")
ax.plot(fig9_res["A_jinf"], color="black", ls="dotted")
ax.plot(fig9_res["A_r0jinf"], color="black", ls="-.")

ax.set_xlim(0, 50)
ax.set_xticks(np.linspace(0, 50, 6))
ax.set_xlabel("T (C)")

ax.set_ylim(-5, 30)
ax.set_yticks(np.linspace(0, 25, 6))
ax.set_ylabel("Assimilation rate, A (μmol m⁻² s⁻¹)")

text_xval = 30
text_yvals = fig9_res.iloc[fig9_res.index.get_indexer([text_xval], method="nearest")[0]]
distance = 1
for I in [500, 250, 100, 0]:
    ax.text(
        x=text_xval,
        y=text_yvals[f"A_{I}"] - distance,
        s=f"{I}",
        va="top",
        ha="center"
    )
    
ax.text(
    x=1,
    y=25,
    s="C=230 µbar\nO=210 mbar",
    va="bottom",
    ha="left"
)

plt.tight_layout()

## Lochocki 2025 figs

In [ ]:
fig3, ax = plt.subplots(
    figsize=(4, 4),
)

data = pd.DataFrame(
    {
        "pco2": np.linspace(1, 800, 100),
    }
)

data["A"] = data["pco2"].apply(lambda x: get_fvcb(pco2=x, model_version="2025", use_2025_default=True)["A"])
data["wc"] = data["pco2"].apply(lambda x: get_fvcb(pco2=x, model_version="2025", use_2025_default=True)["Wc"])
data["wj"] = data["pco2"].apply(lambda x: get_fvcb(pco2=x, model_version="2025", use_2025_default=True)["Wj"])
data["wp"] = data["pco2"].apply(lambda x: get_fvcb(pco2=x, model_version="2025", use_2025_default=True)["Wp"])

data = data.set_index("pco2")

min_identity = data[["wc", "wj", "wp"]].idxmin(axis=1)
switch_mask = (min_identity != min_identity.shift(1)) & min_identity.shift(1).notna()

switch_x = data[switch_mask].index
switch_y = data[["wc", "wj", "wp"]].min(axis=1)[switch_mask]

ax.plot(data["A"], label="A", color="black", lw=7)
ax.plot(data[["wc", "wj", "wp"]].min(axis=1), color="#cccccc", lw=7, label="$W_p$")
ax.plot(data["wc"], color="#bad78c", lw=3, ls="--", label="$W_c$")
ax.plot(data["wj"], color="#f4a17c", lw=3, ls="-.", label="$W_j$")
ax.plot(data["wp"], color="#8e96c3", lw=3, ls=(0, (3, 1, 1, 1, 1, 1)), label="$W_p$")

plt.scatter(switch_x, switch_y, color="white", s=200, zorder=5, marker="*", edgecolor="black")

ax.set_xlim(0, 800)
ax.set_ylim(-20, 50)

ax.set_ylabel("Rates (µmol m⁻² s⁻¹)")
ax.set_xlabel("C (µbar)")

rect_y = 1.05
rect_height = 0.05
ax.add_artist(
    plt.Rectangle(
        xy=(0, rect_y),      # rect_y must also be between 0.0 and 1.0
        width=switch_x[0] / ax.get_xlim()[1],             # 1 means 100% of the plot's width
        height=rect_height,  # rect_height must also be a fraction (e.g., 0.2)
        transform=ax.transAxes,
        color="#bad78c",
        clip_on=False
    )
)
ax.text(
    x=switch_x[0] / ax.get_xlim()[1] / 2,
    y=rect_y + rect_height + 0.1,
    s="$W_c$",
    ha="center",
    va="top",
    fontsize=12,
    transform=ax.transAxes,
    color="#bad78c",
)

ax.add_artist(
    plt.Rectangle(
        xy=(switch_x[0] / ax.get_xlim()[1], rect_y),      # rect_y must also be between 0.0 and 1.0
        width=(switch_x[1] - switch_x[0]) / ax.get_xlim()[1],             # 1 means 100% of the plot's width
        height=rect_height,  # rect_height must also be a fraction (e.g., 0.2)
        transform=ax.transAxes,
        color="#f4a17c",
        clip_on=False
    )
)
ax.text(
    x=((switch_x[1] - switch_x[0]) / 2 + switch_x[0]) / ax.get_xlim()[1],
    y=rect_y + rect_height + 0.1,
    s="$W_j$",
    ha="center",
    va="top",
    fontsize=12,
    transform=ax.transAxes,
    color="#f4a17c",
)
ax.add_artist(
    plt.Rectangle(
        xy=(switch_x[1] / ax.get_xlim()[1], rect_y),      # rect_y must also be between 0.0 and 1.0
        width=(ax.get_xlim()[1] - switch_x[1]) / ax.get_xlim()[1],             # 1 means 100% of the plot's width
        height=rect_height,  # rect_height must also be a fraction (e.g., 0.2)
        transform=ax.transAxes,
        color="#8e96c3",
        clip_on=False
    )
)
ax.text(
    x=((ax.get_xlim()[1] - switch_x[1]) / 2 + switch_x[1]) / ax.get_xlim()[1],
    y=rect_y + rect_height + 0.1,
    s="$W_p$",
    ha="center",
    va="top",
    fontsize=12,
    transform=ax.transAxes,
    color="#8e96c3"
)
ax.legend(frameon=False, title="Min-W variant:")


plt.tight_layout()